Conexion Base de datos Mongo

In [1]:

from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["upme_solar_db"]
print("Conexión exitosa:", client.server_info()["version"])

Conexión exitosa: 8.3.2


In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["upme_solar_db"]
print("Conexión exitosa, versión:", client.server_info()["version"])
print("Colecciones existentes:", db.list_collection_names())

Conexión exitosa, versión: 8.3.2
Colecciones existentes: []


** Verificacion

In [3]:
from pymongo import MongoClient
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client["upme_solar_db"]

Carga municipios pdet a Mongo

In [ ]:
import geopandas as gpd
from pymongo import MongoClient
from datetime import datetime
import os

import sys
from pathlib import Path

# Agregar raíz del proyecto al path para importar config
sys.path.append(str(Path(__file__).resolve().parent.parent.parent))
from config import BASE
client = MongoClient("mongodb://localhost:27017/")
db = client["upme_solar_db"]
col = db["municipios_pdet"]
col.drop()

pdet = gpd.read_file(os.path.join(BASE, "semana_2", "datos", "procesados", "municipios_pdet.geojson"))

docs = []
for _, fila in pdet.iterrows():
    doc = {
        "cod_dane":       str(fila["cod_dane"]),
        "nombre":         fila["Municipio"],
        "departamento":   fila["Departamento"],
        "subregion_pdet": fila["SubPDET"],
        "area_km2":       float(fila["area_km2"]),
        "pdet":           True,
        "geometry":       fila["geometry"].__geo_interface__,
        "fuente":         "MGN2025",
        "cargado_en":     datetime.utcnow()
    }
    docs.append(doc)

col.insert_many(docs)
col.create_index([("geometry", "2dsphere")])
col.create_index([("cod_dane", 1)], unique=True)
print(f"municipios_pdet: {col.count_documents({})} documentos cargados")

Carga Google a Mongo

In [ ]:
import pandas as pd
from pymongo import MongoClient
from shapely import wkt
from datetime import datetime
import os
import sys
from pathlib import Path

# Agregar raíz del proyecto al path para importar config
sys.path.append(str(Path(__file__).resolve().parent.parent.parent))
from config import BASE
client = MongoClient("mongodb://localhost:27017/")
db     = client["upme_solar_db"]
col    = db["goo_buildings"]
col.drop()

csv_path = os.path.join(BASE, "semana_3", "datos", "procesados", "google_pdet_filtrado.csv")

BATCH_SIZE = 10_000
total = 0
errores = 0
batch = []

for chunk in pd.read_csv(csv_path, chunksize=BATCH_SIZE):
    for _, fila in chunk.iterrows():
        try:
            geom = wkt.loads(fila["geometry"])
            doc  = {
                "geometry":           geom.__geo_interface__,
                "area_m2":            float(fila["area_in_meters"]),
                "confidence":         float(fila["confidence"]),
                "cod_dane_municipio": str(fila["cod_dane_municipio"]),
                "fuente":             "google",
                "cargado_en":         datetime.utcnow()
            }
            batch.append(doc)
        except Exception:
            errores += 1
            continue

        if len(batch) >= BATCH_SIZE:
            col.insert_many(batch)
            total += len(batch)
            batch = []
            print(f"  Insertados: {total:,}  |  Errores: {errores}")

if batch:
    col.insert_many(batch)
    total += len(batch)

col.create_index([("geometry", "2dsphere")])
col.create_index([("cod_dane_municipio", 1)])
col.create_index([("confidence", -1)])
print(f"\ngoo_buildings: {col.count_documents({}):,} documentos cargados")
print(f"Errores omitidos: {errores}")

Carga Microsoft a mongo

In [ ]:
import pandas as pd
import json
from pymongo import MongoClient
from datetime import datetime
import os
import sys
from pathlib import Path

# Agregar raíz del proyecto al path para importar config
sys.path.append(str(Path(__file__).resolve().parent.parent.parent))
from config import BASE
client = MongoClient("mongodb://localhost:27017/")
db     = client["upme_solar_db"]
col    = db["ms_buildings"]
col.drop()
print("Colección limpia.")

csv_path   = os.path.join(BASE, "semana_3", "datos", "procesados", "microsoft_pdet_limpio.csv")
BATCH_SIZE = 1_000
total      = 0
errores    = 0
batch      = []

df = pd.read_csv(csv_path)
print(f"Total filas a cargar: {len(df)}")

for _, fila in df.iterrows():
    try:
        geom_str = fila["geometry"]

        # Si viene con escapes dobles, limpiarlos
        if isinstance(geom_str, str):
            geom_str = geom_str.replace('""', '"').strip('"')

        geom = json.loads(geom_str)

        doc = {
            "geometry":           geom,
            "area_m2":            float(fila["area_m2"]),
            "height":             float(fila["height"]),
            "cod_dane_municipio": None,
            "fuente":             "microsoft",
            "cargado_en":         datetime.utcnow()
        }
        batch.append(doc)

    except Exception as e:
        errores += 1
        if errores <= 3:  # mostrar solo los primeros 3 errores para debug
            print(f"  Error fila {_}: {e}")
            print(f"  Valor: {str(fila['geometry'])[:100]}")
        continue

    if len(batch) >= BATCH_SIZE:
        col.insert_many(batch)
        total += len(batch)
        batch = []
        print(f"  Insertados: {total:,}  |  Errores: {errores}")

if batch:
    col.insert_many(batch)
    total += len(batch)

print(f"\nCreando índices...")
col.create_index([("geometry", "2dsphere")])
col.create_index([("cod_dane_municipio", 1)])
print("Índices:", list(col.index_information().keys()))

print(f"\n✅ Carga completada")
print(f"   Total insertados: {total:,}")
print(f"   Errores:          {errores}")
print(f"   Total en MongoDB: {col.count_documents({}):,}")

Verificacion Carga de datos Mongo

In [4]:
print("=" * 50)
print("1. CONTEOS POR COLECCIÓN")
print("=" * 50)
for col in ["municipios_pdet", "ms_buildings", "goo_buildings"]:
    n = db[col].count_documents({})
    print(f"  {col}: {n:,} documentos")

1. CONTEOS POR COLECCIÓN
  municipios_pdet: 170 documentos
  ms_buildings: 5,258 documentos
  goo_buildings: 2,263,446 documentos


In [5]:
print("\n" + "=" * 50)
print("2. ÍNDICES ACTIVOS")
print("=" * 50)
for col in ["municipios_pdet", "ms_buildings", "goo_buildings"]:
    indices = list(db[col].index_information().keys())
    print(f"  {col}: {indices}")


2. ÍNDICES ACTIVOS
  municipios_pdet: ['_id_', 'geometry_2dsphere', 'cod_dane_1']
  ms_buildings: ['_id_', 'geometry_2dsphere', 'cod_dane_municipio_1']
  goo_buildings: ['_id_', 'geometry_2dsphere', 'cod_dane_municipio_1', 'confidence_-1']


In [6]:
print("\n" + "=" * 50)
print("3. ESTADÍSTICAS DE ÁREA (area_m2)")
print("=" * 50)
for col in ["ms_buildings", "goo_buildings"]:
    stats = list(db[col].aggregate([{
        "$group": {
            "_id":          None,
            "total":        {"$sum": 1},
            "area_total":   {"$sum": "$area_m2"},
            "area_promedio":{"$avg": "$area_m2"},
            "area_max":     {"$max": "$area_m2"},
            "area_min":     {"$min": "$area_m2"}
        }
    }]))[0]
    print(f"\n  {col}:")
    print(f"    Total edificios:  {stats['total']:,}")
    print(f"    Área total:       {stats['area_total']:,.0f} m²")
    print(f"    Área promedio:    {stats['area_promedio']:.1f} m²")
    print(f"    Área máxima:      {stats['area_max']:.1f} m²")
    print(f"    Área mínima:      {stats['area_min']:.1f} m²")


3. ESTADÍSTICAS DE ÁREA (area_m2)

  ms_buildings:
    Total edificios:  5,258
    Área total:       702,146 m²
    Área promedio:    133.5 m²
    Área máxima:      3754.7 m²
    Área mínima:      11.4 m²

  goo_buildings:
    Total edificios:  2,263,446
    Área total:       207,447,099 m²
    Área promedio:    91.7 m²
    Área máxima:      20916.7 m²
    Área mínima:      2.5 m²


In [7]:
print("\n" + "=" * 50)
print("4. PRUEBA ESPACIAL — punto en Tibú (municipio PDET)")
print("=" * 50)
punto_tibu = {"type": "Point", "coordinates": [-72.7317, 8.6572]}

mpio = db.municipios_pdet.find_one({
    "geometry": {"$geoIntersects": {"$geometry": punto_tibu}}
})
if mpio:
    print(f"  ✅ Municipio encontrado: {mpio['nombre']} — {mpio['departamento']}")
else:
    print("  ❌ No se encontró municipio — revisar índice")


4. PRUEBA ESPACIAL — punto en Tibú (municipio PDET)
  ✅ Municipio encontrado: TIBÚ — NORTE DE SANTANDER


In [8]:
print("\n" + "=" * 50)
print("5. TOP 5 MUNICIPIOS con más edificios (Google)")
print("=" * 50)
pipeline = [
    {"$match":  {"cod_dane_municipio": {"$ne": None}}},
    {"$group":  {"_id": "$cod_dane_municipio", "total": {"$sum": 1}}},
    {"$sort":   {"total": -1}},
    {"$limit":  5}
]
for r in db.goo_buildings.aggregate(pipeline):
    # Buscar nombre del municipio
    mpio = db.municipios_pdet.find_one({"cod_dane": r["_id"]})
    nombre = mpio["nombre"] if mpio else r["_id"]
    print(f"  {nombre}: {r['total']:,} edificios")


5. TOP 5 MUNICIPIOS con más edificios (Google)
  SANTA MARTA: 154,601 edificios
  VALLEDUPAR: 141,586 edificios
  BUENAVENTURA: 88,341 edificios
  5837: 66,587 edificios
  FLORENCIA: 48,674 edificios


In [9]:
print("\n" + "=" * 50)
print("6. DISTRIBUCIÓN POR CONFIDENCE (Google)")
print("=" * 50)
pipeline = [
    {"$bucket": {
        "groupBy":    "$confidence",
        "boundaries": [0.7, 0.8, 0.9, 1.01],
        "default":    "otro",
        "output":     {"count": {"$sum": 1}}
    }}
]
for r in db.goo_buildings.aggregate(pipeline):
    print(f"  confidence {r['_id']}: {r['count']:,} edificios")

print("\n✅ Verificación completada")


6. DISTRIBUCIÓN POR CONFIDENCE (Google)
  confidence 0.7: 1,146,865 edificios
  confidence 0.8: 1,040,098 edificios
  confidence 0.9: 76,483 edificios

✅ Verificación completada


In [11]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["upme_solar_db"]

# Ver cuántos documentos tienen cod_dane_municipio sin ceros
problemas = db.goo_buildings.count_documents({
    "cod_dane_municipio": {"$regex": "^[0-9]{4}$"}
})
print(f"Documentos con código de 4 dígitos: {problemas:,}")

# Corregir — agregar cero al inicio
if problemas > 0:
    db.goo_buildings.update_many(
        {"cod_dane_municipio": {"$regex": "^[0-9]{4}$"}},
        [{"$set": {"cod_dane_municipio": {
            "$concat": ["0", "$cod_dane_municipio"]
        }}}]
    )
    print("Códigos corregidos.")

    # Verificar corrección
    mpio = db.municipios_pdet.find_one({"cod_dane": "05837"})
    if mpio:
        print(f"Municipio 05837: {mpio['nombre']}")
    else:
        print("Municipio 05837: No encontrado")
else:
    print("No hay códigos con 4 dígitos — todo bien.")

Documentos con código de 4 dígitos: 309,598
Códigos corregidos.
Municipio 05837: TURBO


In [12]:
pipeline = [
    {"$match":  {"cod_dane_municipio": {"$ne": None}}},
    {"$group":  {"_id": "$cod_dane_municipio", "total": {"$sum": 1}}},
    {"$sort":   {"total": -1}},
    {"$limit":  5}
]
print("TOP 5 MUNICIPIOS con más edificios (Google) — corregido:")
for r in db.goo_buildings.aggregate(pipeline):
    mpio = db.municipios_pdet.find_one({"cod_dane": r["_id"]})
    nombre = mpio["nombre"] if mpio else f"SIN NOMBRE ({r['_id']})"
    print(f"  {nombre}: {r['total']:,} edificios")

# Verificar que no quedan códigos de 4 dígitos
restantes = db.goo_buildings.count_documents({
    "cod_dane_municipio": {"$regex": "^[0-9]{4}$"}
})
print(f"\nCódigos de 4 dígitos restantes: {restantes}")

TOP 5 MUNICIPIOS con más edificios (Google) — corregido:
  SANTA MARTA: 154,601 edificios
  VALLEDUPAR: 141,586 edificios
  BUENAVENTURA: 88,341 edificios
  TURBO: 66,587 edificios
  FLORENCIA: 48,674 edificios

Códigos de 4 dígitos restantes: 0


In [ ]:
import pandas as pd
import pandas as pd
import json
from pymongo import MongoClient
from datetime import datetime
import os
import sys
from pathlib import Path
client = MongoClient("mongodb://localhost:27017/")
db = client["upme_solar_db"]

google_df = pd.DataFrame(
    list(db.goo_buildings.find({}, {"_id": 0}))
)
google_df.head()
google_df.columns
google_df.columns.tolist()

NameError: name 'db' is not defined